# ViT-B/16 Multilabel UCMerced — Stratified vs Random Split

## Hipótesis
Los modelos ViT pre-entrenados (ImageNet, ~86 M parámetros) son suficientemente robustos al desbalance de clases como para **no** degradarse significativamente cuando el split train/val es aleatorio en lugar de estratificado.
- **H0 (hipótesis verdadera):** Δ métricas entre ambos modelos < 2 pp → el pre-entrenamiento compensa el desbalance.
- **H1 (hipótesis falsa):** Δ métricas ≥ 2 pp → el split estratificado sigue siendo necesario.

## Protocolo experimental — test compartido

Para aislar el efecto del tipo de split sobre el *entrenamiento*, el **test set es idéntico para ambos modelos** (extraído con split estratificado). Solo varía cómo se divide el pool restante en train/val.

```
2100 imágenes totales
├── TEST COMPARTIDO  (315, estratificado) ← igual para Modelo A y B
└── 1785 restantes
    ├── Modelo A → train_A estratificado (1260) + val_A estratificado (315)
    └── Modelo B → train_B aleatorio     (1260) + val_B aleatorio     (315)
```

| | Modelo A | Modelo B |
|---|---|---|
| Split **train/val** | **Stratified** (MultilabelStratifiedShuffleSplit) | **Random** (train_test_split sin estratificación) |
| Split **test** | Estratificado — **compartido** ✅ | Estratificado — **compartido** ✅ |
| Backbone | ViT-B/16 ImageNet1K | ViT-B/16 ImageNet1K |
| Seed | 42 | 42 |
| LR / WD | 3e-5 / 0.05 | 3e-5 / 0.05 |
| Épocas máx. | 30 | 30 |
| Umbral | 0.5 | 0.5 |

## Métricas reportadas
macro-F1 · micro-F1 · samples-F1 · macro-mAP · Hamming loss · Exact-match accuracy · per-class AP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())

In [ ]:
# Instalar dependencias
!pip install -q torch torchvision lightning matplotlib seaborn scikit-learn \
             scikit-image wandb iterative-stratification torchmetrics

In [ ]:
# Sincronizar repositorio
REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull
%cd {PROJECT_DIR}
print("Working in:", os.getcwd())

## 1. Descarga del dataset UCMerced

In [ ]:
import zipfile, subprocess, shutil

if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')
    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zf:
        zf.extractall('UCMImages')
    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
    os.chdir(PROJECT_DIR)

print(os.listdir('.'))

## 2. Importaciones y semilla global

In [ ]:
import random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, average_precision_score, hamming_loss,
    classification_report
)

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

import torchmetrics

warnings.filterwarnings("ignore")
L.seed_everything(42, workers=True)

DEVICE   = "gpu" if torch.cuda.is_available() else "cpu"
SEED     = 42
Path("outputs/checkpoints").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)

print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Dataset UCMerced Multilabel

In [ ]:
class UCMMultilabelDataset(Dataset):
    """UCM multilabel land-use dataset con 17 clases."""

    def __init__(self, root_dir="ucmdata", label_file="LandUse_Multilabeled.txt",
                 transform=None, image_ext=".tif"):
        self.root_dir   = root_dir
        self.images_dir = os.path.join(root_dir, "Images")
        self.transform  = transform
        self.image_ext  = image_ext

        label_path = os.path.join(root_dir, label_file)
        self.class_names, self.label_matrix = self._parse_labels(label_path)
        self.num_classes  = len(self.class_names)
        self.image_paths  = self._collect_image_paths()

        assert len(self.image_paths) == len(self.label_matrix), (
            f"Mismatch: {len(self.image_paths)} images vs {len(self.label_matrix)} rows.")

    def _parse_labels(self, label_path):
        with open(label_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        class_names = lines[0].split("\t")[1:]
        rows = [list(map(int, l.split("\t")[1:])) for l in lines[1:]]
        return class_names, torch.tensor(rows, dtype=torch.float32)

    def _collect_image_paths(self):
        paths = []
        for subfolder in sorted(e.name for e in os.scandir(self.images_dir)):
            folder = os.path.join(self.images_dir, subfolder)
            for fname in sorted(f for f in os.listdir(folder)
                                if f.lower().endswith(self.image_ext)):
                paths.append(os.path.join(folder, fname))
        return paths

    def __len__(self):  return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.label_matrix[idx]

    def get_class_names(self):   return self.class_names
    def get_class_weights(self):
        pos = self.label_matrix.sum(0).clamp(min=1)
        neg = (len(self) - self.label_matrix.sum(0)).clamp(min=1)
        return neg / pos

## 4. Augmentaciones (idénticas para ambos modelos)

In [ ]:
IMAGE_SIZE = (224, 224)
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_transforms(split="train"):
    if split == "train":
        return transforms.Compose([
            transforms.Resize(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(45),
            transforms.ColorJitter(brightness=0.2, contrast=0.4, saturation=0.3),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ])
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])

## 5. Funciones de split: diseño con test compartido

**Paso 1:** Se extrae el test set con split **estratificado** (mismo para ambos modelos).  
**Paso 2A:** Las imágenes restantes se dividen con split **estratificado** → train/val del Modelo A.  
**Paso 2B:** Las mismas imágenes restantes se dividen con split **aleatorio** → train/val del Modelo B.  

Así, cualquier diferencia de métricas refleja exclusivamente el efecto del tipo de split sobre los datos de **entrenamiento**, no sobre la evaluación.

In [ ]:
def _make_dataloaders(full_ds, train_idx, val_idx, test_idx,
                      batch_size, num_workers):
    """Crea DataLoaders a partir de índices pre-calculados."""
    root = full_ds.root_dir
    lf   = "LandUse_Multilabeled.txt"
    ext  = full_ds.image_ext

    def make_subset(split_name, indices):
        ds = UCMMultilabelDataset(root_dir=root, label_file=lf,
                                  transform=get_transforms(split_name),
                                  image_ext=ext)
        return Subset(ds, indices)

    kw = dict(batch_size=batch_size, num_workers=num_workers,
              pin_memory=torch.cuda.is_available())
    return (
        DataLoader(make_subset("train", train_idx), shuffle=True,  **kw),
        DataLoader(make_subset("val",   val_idx),   shuffle=False, **kw),
        DataLoader(make_subset("test",  test_idx),  shuffle=False, **kw),
    )


def build_shared_test(root_dir="ucmdata", label_file="LandUse_Multilabeled.txt",
                      test_frac=0.15, seed=42, image_ext=".tif"):
    """
    Paso 1 — extrae el test set COMPARTIDO con split estratificado.
    Devuelve (full_ds, tv_idx, test_idx, class_names, Y_all)
    tv_idx = pool de imágenes disponibles para train/val de ambos modelos.
    """
    full_ds = UCMMultilabelDataset(root_dir, label_file, transform=None,
                                   image_ext=image_ext)
    n = len(full_ds)
    Y = full_ds.label_matrix.numpy()

    spl = MultilabelStratifiedShuffleSplit(n_splits=1,
                                           test_size=test_frac,
                                           random_state=seed)
    tv_idx, test_idx = next(spl.split(np.zeros(n), Y))
    return full_ds, tv_idx, test_idx, full_ds.get_class_names(), Y


def build_stratified(full_ds, tv_idx, test_idx, Y,
                     batch_size=32, num_workers=2,
                     val_frac_of_tv=0.15 / 0.85, seed=42):
    """
    Paso 2A — divide tv_idx en train/val con split ESTRATIFICADO.
    val_frac_of_tv = fracción de validación sobre el pool train+val
    (por defecto 0.15/0.85 ≈ 0.1765 para que val sea ~15% del total).
    """
    spl = MultilabelStratifiedShuffleSplit(n_splits=1,
                                           test_size=val_frac_of_tv,
                                           random_state=seed)
    tr_local, val_local = next(spl.split(np.zeros(len(tv_idx)), Y[tv_idx]))
    train_idx = tv_idx[tr_local]
    val_idx   = tv_idx[val_local]

    # pos_weight calculado sobre los índices de entrenamiento de ESTE split
    Y_train = torch.tensor(Y[train_idx], dtype=torch.float32)
    pos     = Y_train.sum(0).clamp(min=1)
    neg     = (len(train_idx) - Y_train.sum(0)).clamp(min=1)
    pos_w   = neg / pos

    loaders = _make_dataloaders(full_ds, train_idx, val_idx, test_idx,
                                batch_size, num_workers)
    return (*loaders, pos_w,
            {"train": train_idx, "val": val_idx, "test": test_idx})


def build_random(full_ds, tv_idx, test_idx, Y,
                 batch_size=32, num_workers=2,
                 val_frac_of_tv=0.15 / 0.85, seed=42):
    """
    Paso 2B — divide tv_idx en train/val con split ALEATORIO puro.
    Mismos tamaños que build_stratified para comparación justa.
    """
    train_idx, val_idx = train_test_split(
        tv_idx, test_size=val_frac_of_tv,
        random_state=seed, shuffle=True
    )

    Y_train = torch.tensor(Y[train_idx], dtype=torch.float32)
    pos     = Y_train.sum(0).clamp(min=1)
    neg     = (len(train_idx) - Y_train.sum(0)).clamp(min=1)
    pos_w   = neg / pos

    loaders = _make_dataloaders(full_ds, train_idx, val_idx, test_idx,
                                batch_size, num_workers)
    return (*loaders, pos_w,
            {"train": train_idx, "val": val_idx, "test": test_idx})


## 6. Análisis de distribución de clases por tipo de split

Como el test set es **compartido**, la comparación relevante es la distribución de clases en el **conjunto de entrenamiento**.  
Una diferencia marcada entre ambas distribuciones evidencia el desbalance inducido por el split aleatorio.

In [ ]:
BATCH, WORKERS = 32, 2

# ── Paso 1: test set compartido ─────────────────────────────────────────────
full_ds, tv_idx, test_idx_shared, classes, Y_all = build_shared_test()
NUM_CLASSES = len(classes)

# ── Paso 2A: split estratificado (train/val) ─────────────────────────────────
(tr_s, va_s, te_shared_s,
 pos_w_s, splits_s) = build_stratified(full_ds, tv_idx, test_idx_shared, Y_all,
                                        batch_size=BATCH, num_workers=WORKERS)

# ── Paso 2B: split aleatorio (train/val) ─────────────────────────────────────
(tr_r, va_r, te_shared_r,
 pos_w_r, splits_r) = build_random(full_ds, tv_idx, test_idx_shared, Y_all,
                                    batch_size=BATCH, num_workers=WORKERS)

# te_shared_s y te_shared_r apuntan exactamente a los mismos índices
assert np.array_equal(sorted(splits_s['test']), sorted(splits_r['test'])), \
    '❌ ERROR: los test sets no coinciden'
print('✅ Test sets verificados: idénticos para ambos modelos')
print(f'   Train: {len(splits_s["train"])} | Val: {len(splits_s["val"])} | Test: {len(splits_s["test"])}')

# ── Distribución de clases ───────────────────────────────────────────────────
def class_freq(indices):
    return Y_all[indices].sum(axis=0)

freq_train_s = class_freq(splits_s['train'])
freq_train_r = class_freq(splits_r['train'])
freq_test    = class_freq(splits_s['test'])   # mismo para A y B

x = np.arange(NUM_CLASSES)
w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Izquierda: train A vs train B
axes[0].bar(x - w/2, freq_train_s, width=w, label='Stratified (A)', color='steelblue', alpha=0.85)
axes[0].bar(x + w/2, freq_train_r, width=w, label='Random (B)',     color='tomato',    alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('# positive samples'); axes[0].set_title('Training set — frecuencia por clase')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Derecha: test compartido
axes[1].bar(x, freq_test, color='mediumseagreen', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('# positive samples')
axes[1].set_title('Test set COMPARTIDO (estratificado) — mismo para A y B')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Distribución de clases — train (A vs B) y test compartido', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/class_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla de diferencia en el TRAIN set
diff_df = pd.DataFrame({
    'class':       classes,
    'train_strat': freq_train_s.astype(int),
    'train_random':freq_train_r.astype(int),
    'abs_diff':    np.abs(freq_train_s - freq_train_r).astype(int),
}).sort_values('abs_diff', ascending=False)
print('\nDiferencia absoluta en el TRAIN set (clases más afectadas por random split):')
print(diff_df.to_string(index=False))


## 7. Arquitectura ViT-B/16

Se reemplaza únicamente la cabeza de clasificación (`model.heads.head`) por una nueva capa `Linear(768 → 17)`.  
Todos los pesos del transformer permanecen entrenables (fine-tuning completo).

In [ ]:
def build_vit_b16(num_classes: int):
    """ViT-B/16 pre-entrenado en ImageNet1K con nueva cabeza multilabel."""
    weights = tvm.ViT_B_16_Weights.IMAGENET1K_V1
    model   = tvm.vit_b_16(weights=weights)
    in_feat = model.heads.head.in_features          # 768
    model.heads.head = nn.Linear(in_feat, num_classes)
    return model

# Verificación rápida (sin inicializar para no desperdiciar VRAM)
_tmp = build_vit_b16(NUM_CLASSES)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"ViT-B/16  |  parámetros entrenables: {n_params:,}  |  salidas: {NUM_CLASSES}")
del _tmp

## 8. LightningModule — multilabel con BCEWithLogitsLoss + pos_weight

In [ ]:
class LitViTMultilabel(L.LightningModule):
    def __init__(self, model, num_classes, lr=3e-5, weight_decay=5e-2,
                 max_epochs=30, threshold=0.5, pos_weight=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model     = model
        pw = torch.as_tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

        mk = dict(task="multilabel", num_labels=num_classes, threshold=threshold)
        self.train_f1  = torchmetrics.F1Score(average="macro", **mk)
        self.val_f1    = torchmetrics.F1Score(average="macro", **mk)
        self.test_f1   = torchmetrics.F1Score(average="macro", **mk)
        self.val_acc   = torchmetrics.Accuracy(average="macro", **mk)
        self.test_acc  = torchmetrics.Accuracy(average="macro", **mk)
        self.val_map   = torchmetrics.AveragePrecision(
            task="multilabel", num_labels=num_classes, average="macro")
        self.test_map  = torchmetrics.AveragePrecision(
            task="multilabel", num_labels=num_classes, average="macro")

    def forward(self, x):  return self.model(x)

    def _step(self, batch):
        imgs, labels = batch
        logits = self(imgs)
        return self.criterion(logits, labels), logits, labels

    def training_step(self, batch, _):
        loss, logits, labels = self._step(batch)
        self.train_f1.update(logits, labels.int())
        self.log("train_loss", loss,          on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_f1",   self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        loss, logits, labels = self._step(batch)
        self.val_f1.update(logits, labels.int())
        self.val_acc.update(logits, labels.int())
        self.val_map.update(logits, labels.int())
        self.log("val_loss", loss,         on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_f1",   self.val_f1,  on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc",  self.val_acc, on_step=False, on_epoch=True)
        self.log("val_map",  self.val_map, on_step=False, on_epoch=True)

    def test_step(self, batch, _):
        loss, logits, labels = self._step(batch)
        self.test_f1.update(logits, labels.int())
        self.test_acc.update(logits, labels.int())
        self.test_map.update(logits, labels.int())
        self.log("test_loss", loss,          on_step=False, on_epoch=True)
        self.log("test_f1",   self.test_f1,  on_step=False, on_epoch=True)
        self.log("test_acc",  self.test_acc, on_step=False, on_epoch=True)
        self.log("test_map",  self.test_map, on_step=False, on_epoch=True)

    def predict_step(self, batch, _):
        imgs, labels = batch
        logits = self(imgs)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= self.hparams.threshold).int()
        return {"probs": probs, "preds": preds, "labels": labels.int()}

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(),
                                lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.hparams.max_epochs)
        return [opt], [sched]

## 9. Hiperparámetros comunes

In [ ]:
MAX_EPOCHS   = 30     # Más épocas permiten mejor convergencia
LR           = 3e-5   # LR reducido — recomendado para fine-tuning de ViT
WEIGHT_DECAY = 5e-2   # Weight decay estándar para ViT (Dosovitskiy et al.)
THRESHOLD    = 0.5    # Umbral de decisión multilabel
PATIENCE     = 7      # Early stopping

print(f"MAX_EPOCHS={MAX_EPOCHS} | LR={LR} | WD={WEIGHT_DECAY} | THRESHOLD={THRESHOLD}")
print(f"NUM_CLASSES={NUM_CLASSES} | BATCH={BATCH} | SEED={SEED}")

## 10. Entrenamiento — Modelo A (Stratified Split)

In [ ]:
print("=" * 60)
print("  MODELO A — STRATIFIED SPLIT")
print("=" * 60)

backbone_A = build_vit_b16(NUM_CLASSES)

lit_A = LitViTMultilabel(
    model=backbone_A, num_classes=NUM_CLASSES,
    lr=LR, weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS, threshold=THRESHOLD,
    pos_weight=pos_w_s,
)

ckpt_A = ModelCheckpoint(
    dirpath="outputs/checkpoints",
    filename="vitA_stratified-best-{epoch:02d}-{val_f1:.4f}",
    monitor="val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_A = EarlyStopping(monitor="val_f1", mode="max",
                        patience=PATIENCE, min_delta=1e-3, verbose=True)
logger_A = CSVLogger("outputs/logs", name="vitA_stratified")

trainer_A = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto", devices="auto",
    callbacks=[ckpt_A, early_A],
    logger=logger_A,
    log_every_n_steps=10,
    deterministic=True,
)
trainer_A.fit(lit_A, train_dataloaders=tr_s, val_dataloaders=va_s)
print(f"\nMejor checkpoint A: {ckpt_A.best_model_path}")

## 11. Entrenamiento — Modelo B (Random Split)

In [ ]:
print("=" * 60)
print("  MODELO B — RANDOM SPLIT")
print("=" * 60)

L.seed_everything(SEED, workers=True)   # misma semilla → misma inicialización

backbone_B = build_vit_b16(NUM_CLASSES)

lit_B = LitViTMultilabel(
    model=backbone_B, num_classes=NUM_CLASSES,
    lr=LR, weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS, threshold=THRESHOLD,
    pos_weight=pos_w_r,
)

ckpt_B = ModelCheckpoint(
    dirpath="outputs/checkpoints",
    filename="vitB_random-best-{epoch:02d}-{val_f1:.4f}",
    monitor="val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_B = EarlyStopping(monitor="val_f1", mode="max",
                        patience=PATIENCE, min_delta=1e-3, verbose=True)
logger_B = CSVLogger("outputs/logs", name="vitB_random")

trainer_B = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto", devices="auto",
    callbacks=[ckpt_B, early_B],
    logger=logger_B,
    log_every_n_steps=10,
    deterministic=True,
)
trainer_B.fit(lit_B, train_dataloaders=tr_r, val_dataloaders=va_r)
print(f"\nMejor checkpoint B: {ckpt_B.best_model_path}")

## 12. Evaluación en test — ambos modelos

In [ ]:
def evaluate_model(trainer, lit_model, ckpt_cb, test_loader, label=''):
    """Carga el mejor checkpoint y devuelve predicciones + métricas sklearn."""
    print(f"\n{'─'*50}\n  Evaluando: {label}\n{'─'*50}")
    trainer.test(lit_model, dataloaders=test_loader, ckpt_path='best')

    best_path = ckpt_cb.best_model_path
    lit_model = LitViTMultilabel.load_from_checkpoint(
        best_path, model=lit_model.model)
    preds_out = trainer.predict(lit_model, dataloaders=test_loader)

    probs  = torch.cat([b['probs']  for b in preds_out]).cpu().numpy()
    preds  = torch.cat([b['preds']  for b in preds_out]).cpu().numpy()
    labels = torch.cat([b['labels'] for b in preds_out]).cpu().numpy()

    metrics = {
        'macro_f1':    f1_score(labels, preds, average='macro',   zero_division=0),
        'micro_f1':    f1_score(labels, preds, average='micro',   zero_division=0),
        'samples_f1':  f1_score(labels, preds, average='samples', zero_division=0),
        'macro_mAP':   average_precision_score(labels, probs, average='macro'),
        'hamming':     hamming_loss(labels, preds),
        'exact_match': (preds == labels).all(axis=1).mean(),
        'per_class_f1': f1_score(labels, preds, average=None, zero_division=0),
        'per_class_ap': average_precision_score(labels, probs, average=None),
    }
    print(f"  macro-F1    = {metrics['macro_f1']:.4f}")
    print(f"  micro-F1    = {metrics['micro_f1']:.4f}")
    print(f"  samples-F1  = {metrics['samples_f1']:.4f}")
    print(f"  macro-mAP   = {metrics['macro_mAP']:.4f}")
    print(f"  Hamming     = {metrics['hamming']:.4f}")
    print(f"  Exact-match = {metrics['exact_match']:.4f}")
    return metrics, probs, preds, labels


# Ambos modelos se evalúan sobre el MISMO test set (te_shared_s == te_shared_r)
results_A, probs_A, preds_A, labels_A = evaluate_model(
    trainer_A, lit_A, ckpt_A, te_shared_s, label='Modelo A — Stratified')

results_B, probs_B, preds_B, labels_B = evaluate_model(
    trainer_B, lit_B, ckpt_B, te_shared_r, label='Modelo B — Random')

# Verificación: labels_A y labels_B deben ser idénticos (mismo test set)
assert np.array_equal(labels_A, labels_B), \
    '❌ ERROR: labels de test difieren entre modelos'
print('\n✅ Verificado: ambos modelos fueron evaluados con las mismas etiquetas de test.')


## 13. Tabla comparativa de métricas

In [ ]:
metric_names = ["macro_f1", "micro_f1", "samples_f1", "macro_mAP", "hamming", "exact_match"]
labels_nice  = ["Macro-F1 ↑", "Micro-F1 ↑", "Samples-F1 ↑",
                "Macro-mAP ↑", "Hamming Loss ↓", "Exact-Match ↑"]

rows = []
for mn, ln in zip(metric_names, labels_nice):
    a = results_A[mn]
    b = results_B[mn]
    delta = b - a if "hamming" not in mn else a - b   # Hamming: menor = mejor
    rows.append({"Métrica": ln,
                 "Modelo A (Stratified)": round(float(a), 4),
                 "Modelo B (Random)":     round(float(b), 4),
                 "Δ (B − A)": round(float(delta), 4)})

comp_df = pd.DataFrame(rows)
comp_df.to_csv("outputs/metrics_comparison.csv", index=False)
print(comp_df.to_string(index=False))

# ── Evaluación de la hipótesis ───────────────────────────────────────────────
max_delta = comp_df["Δ (B − A)"].abs().max()
THRESHOLD_H = 0.02   # 2 pp
print(f"\nΔ máxima observada: {max_delta:.4f}  (umbral hipótesis: {THRESHOLD_H})")
if max_delta < THRESHOLD_H:
    print("✅  HIPÓTESIS VERDADERA: El pre-entrenamiento de ViT compensa el desbalance.")
    print("   La diferencia entre split estratificado y aleatorio es marginal (< 2 pp).")
else:
    print("❌  HIPÓTESIS FALSA: El tipo de split sí afecta significativamente las métricas.")
    print("   El split estratificado sigue siendo necesario para este dataset/modelo.")

## 14. Curvas de aprendizaje comparativas

In [ ]:
def load_metrics_csv(log_dir_base, name):
    """Carga el CSV de métricas del primer versión del logger."""
    csv_path = Path(log_dir_base) / name / "version_0" / "metrics.csv"
    df = pd.read_csv(csv_path)
    return df.groupby("epoch").mean(numeric_only=True).reset_index()

log_base = "outputs/logs"
df_A = load_metrics_csv(log_base, "vitA_stratified")
df_B = load_metrics_csv(log_base, "vitB_random")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

pairs = [
    (axes[0, 0], "train_loss", "val_loss",  "BCE Loss",  "Loss (entrenamiento y validación)"),
    (axes[0, 1], "train_f1",  "val_f1",    "Macro-F1",  "Macro-F1 (entrenamiento y validación)"),
    (axes[1, 0], "val_f1",    None,         "Val Macro-F1", "Macro-F1 en validación — A vs B"),
    (axes[1, 1], "val_map",   None,         "Val Macro-mAP","Macro-mAP en validación — A vs B"),
]

for ax, col_train, col_val, ylabel, title in pairs:
    if col_val is not None:   # curvas train vs val de UN modelo
        for df, label_prefix, color in [
            (df_A, "Stratified", "steelblue"),
            (df_B, "Random",     "tomato"),
        ]:
            if col_train in df:
                ax.plot(df["epoch"], df[col_train],
                        ls="--", color=color, alpha=0.6, label=f"{label_prefix} train")
            if col_val in df:
                ax.plot(df["epoch"], df[col_val],
                        ls="-",  color=color, lw=2, label=f"{label_prefix} val")
    else:                      # comparación de la misma métrica entre A y B
        for df, label, color in [
            (df_A, "Stratified (A)", "steelblue"),
            (df_B, "Random (B)",     "tomato"),
        ]:
            if col_train in df:
                ax.plot(df["epoch"], df[col_train],
                        lw=2, color=color, label=label)

    ax.set_xlabel("Época"); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Curvas de aprendizaje — ViT-B/16: Stratified vs Random",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/learning_curves_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Métricas per-class — F1 y AP por clase

In [ ]:
# labels_A == labels_B (mismo test set), usamos labels_A como referencia
positives = labels_A.sum(axis=0).astype(int)

perclass_df = pd.DataFrame({
    'class':         classes,
    'n_pos_test':    positives,           # igual para ambos
    'F1_stratified': results_A['per_class_f1'].round(4),
    'F1_random':     results_B['per_class_f1'].round(4),
    'ΔF1 (R-S)':    (results_B['per_class_f1'] - results_A['per_class_f1']).round(4),
    'AP_stratified': results_A['per_class_ap'].round(4),
    'AP_random':     results_B['per_class_ap'].round(4),
    'ΔAP (R-S)':    (results_B['per_class_ap'] - results_A['per_class_ap']).round(4),
}).sort_values('ΔF1 (R-S)')

perclass_df.to_csv('outputs/per_class_comparison.csv', index=False)
print(perclass_df.to_string(index=False))

# ── Gráfico per-class ──────────────────────────────────────────────────────
order = np.argsort(results_A['per_class_f1'])
y = np.arange(NUM_CLASSES)
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
titles = ['Per-class F1', 'Per-class AP (mAP)']
keys   = [('per_class_f1', 'per_class_f1'), ('per_class_ap', 'per_class_ap')]

for ax, title, (key_a, key_b) in zip(axes, titles, keys):
    ax.barh(y - w/2, results_A[key_a][order], height=w,
            label='Stratified (A)', color='steelblue', alpha=0.85)
    ax.barh(y + w/2, results_B[key_b][order], height=w,
            label='Random (B)',     color='tomato',    alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels([classes[i] for i in order], fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Score'); ax.set_title(title)
    ax.legend(); ax.grid(axis='x', alpha=0.3)

plt.suptitle('Comparación per-class: Stratified vs Random (test set compartido)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/per_class_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 16. Heatmap de diferencia ΔF1 (Random − Stratified) por clase

In [ ]:
delta_f1 = results_B["per_class_f1"] - results_A["per_class_f1"]
delta_ap = results_B["per_class_ap"] - results_A["per_class_ap"]

hm_data = np.vstack([delta_f1, delta_ap]).T

fig, ax = plt.subplots(figsize=(5, max(5, 0.4 * NUM_CLASSES)))
im = ax.imshow(hm_data, cmap="RdBu", vmin=-0.15, vmax=0.15, aspect="auto")
ax.set_xticks([0, 1])
ax.set_xticklabels(["ΔF1", "ΔAP"], fontsize=11)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(classes, fontsize=9)
for i in range(NUM_CLASSES):
    for j, val in enumerate([delta_f1[i], delta_ap[i]]):
        ax.text(j, i, f"{val:+.3f}", ha="center", va="center", fontsize=8,
                color="black" if abs(val) < 0.08 else "white")
plt.colorbar(im, ax=ax, label="Δ (Random − Stratified)")
ax.set_title("Impacto del tipo de split por clase\n(azul = stratified mejor | rojo = random mejor)",
             fontsize=11)
plt.tight_layout()
plt.savefig("outputs/delta_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 17. Conclusión experimental

Este bloque imprime el veredicto final sobre la hipótesis de trabajo.

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║           RESULTADO EXPERIMENTAL — HIPÓTESIS ViT             ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Modelo A (Stratified)  macro-F1 = {results_A['macro_f1']:.4f}                ║")
print(f"║  Modelo B (Random)      macro-F1 = {results_B['macro_f1']:.4f}                ║")
print(f"║  Δ macro-F1             =  {results_B['macro_f1'] - results_A['macro_f1']:+.4f}               ║")
print("╠══════════════════════════════════════════════════════════════╣")
max_d = comp_df["Δ (B − A)"].abs().max()
if max_d < 0.02:
    verdict = "HIPÓTESIS VERDADERA  ✅"
    detail  = ("La diferencia máxima entre ambos modelos es < 2 pp.\n"
               "  ViT pre-entrenado en ImageNet compensa el desbalance\n"
               "  de clases presente en UC-Merced.")
else:
    verdict = "HIPÓTESIS FALSA  ❌"
    detail  = (f"La diferencia máxima es {max_d:.4f} (> 2 pp).\n"
               "  El split estratificado sigue siendo relevante.\n"
               "  El pre-entrenamiento no es suficiente para compensar\n"
               "  el sesgo de muestreo aleatorio.")
print(f"║  Veredicto: {verdict:<50}║")
print("╠══════════════════════════════════════════════════════════════╣")
for line in detail.split("\n"):
    print(f"║  {line:<60}║")
print("╚══════════════════════════════════════════════════════════════╝")

---
## 18. Referencias bibliográficas

---

### [1] Arquitectura original Vision Transformer (ViT)
Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., ... & Houlsby, N. (2021).  
**An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.**  
*International Conference on Learning Representations (ICLR 2021).*  
DOI / arXiv: https://doi.org/10.48550/arXiv.2010.11929

---

### [2] Aprendizaje con datos desbalanceados en ViT
Xu, Z., Liu, R., Yang, S., Chai, Z., & Yuan, C. (2023).  
**Learning Imbalanced Data with Vision Transformers.**  
*IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR 2023).*  
DOI / arXiv: https://doi.org/10.48550/arXiv.2212.02015

> **Relevancia directa:** Demuestra que la degradación de ViT ante datos desbalanceados es mayor que en CNNs, pero puede mitigarse con reweighting de la pérdida — exactamente lo que hace `BCEWithLogitsLoss(pos_weight=...)` en este notebook.

---

### [3] Dataset UCMerced multilabel — referencia original
Yang, Y., & Newsam, S. (2010).  
**Bag-of-visual-words and spatial extensions for land-use classification.**  
*Proceedings of the 18th SIGSPATIAL International Conference on Advances in Geographic Information Systems.*  
DOI: https://doi.org/10.1145/1869790.1869829

---

### [4] Clasificación multilabel con CNN en UCMerced + data augmentation
Chaudhuri, B., Demir, B., Chaudhuri, S., & Bruzzone, L. (2019).  
**Multilabel remote sensing image retrieval using a semi-supervised graph-theoretic method.**  
*IEEE Transactions on Geoscience and Remote Sensing, 56*(2), 1100–1112.  
DOI: https://doi.org/10.1109/TGRS.2017.2760909

---

### [5] Deep learning para clasificación multilabel de cobertura terrestre
Lobry, S., Tuia, D., & Wegner, J. D. (2019).  
**Deep Learning for Multilabel Land Cover Scene Categorization Using Data Augmentation.**  
*IEEE Geoscience and Remote Sensing Letters, 16*(7), 1082–1086.  
DOI: https://doi.org/10.1109/LGRS.2019.2891904

> **Relevancia:** Usa exactamente el dataset UCMerced multilabel (LandUse_Multilabeled.txt) que se emplea en este notebook. Establece baselines de CNN para comparación.

---

### [6] Stratified split multilabel — métricas de calidad
Sechidis, K., Tsoumakas, G., & Vlahavas, I. (2011).  
**On the stratification of multi-label data.**  
*Machine Learning and Knowledge Discovery in Databases (ECML PKDD 2011).*  
DOI: https://doi.org/10.1007/978-3-642-23808-6_10

> **Relevancia:** Algoritmo base detrás de `iterative-stratification` (MultilabelStratifiedShuffleSplit). Justifica por qué el split estratificado preserva mejor la distribución de etiquetas.

---

### [7] Iterative stratification para datos multilabel
Szymański, P., & Kajdanowicz, T. (2017).  
**A network perspective on stratification of multi-label data.**  
*Proceedings of the First International Workshop on Learning with Imbalanced Domains.*  
DOI / arXiv: https://doi.org/10.48550/arXiv.1704.05634

> **Relevancia:** Paper detrás de la librería `iterative-stratification` usada en este notebook.

---

### [8] Revisión contemporánea sobre class imbalance en visión por computador
Santos, M. S., Abreu, P. H., & Japkowicz, N. (2023).  
**Tackling class imbalance in computer vision: a contemporary review.**  
*Artificial Intelligence Review, 57*(1), 1–65.  
DOI: https://doi.org/10.1007/s10462-023-10557-6

---

### [9] Transformers para clasificación multilabel en teledetección
He, N., Fang, L., & Plaza, A. (2023).  
**Multi-label remote sensing classification with self-supervised gated multi-modal transformers.**  
*Remote Sensing, 15*(14), 3511.  
DOI: https://doi.org/10.3390/rs15143511

---

### [10] Pre-entrenamiento de ViT en teledetección (RSP)
Wang, D., Zhang, J., Du, B., Xia, G. S., & Tao, D. (2022).  
**An Empirical Study of Remote Sensing Pretraining.**  
*IEEE Transactions on Geoscience and Remote Sensing, 61*, 1–20.  
DOI: https://doi.org/10.1109/TGRS.2022.3216603

> **Relevancia:** Demuestra que modelos ViT pre-entrenados en grandes corpus de imágenes de satélite superan a ResNet en múltiples benchmarks de teledetección, incluyendo transfer learning a datasets pequeños como UCMerced.

---

### [11] Fine-tuning de modelos de visión con SGD y comparación de estrategias
Kumar, A., Raghunathan, A., Jones, R., Ma, T., & Liang, P. (2022).  
**Fine-Tuning can Distort Pretrained Features and Underperform Out-of-Distribution.**  
*International Conference on Learning Representations (ICLR 2022).*  
DOI / arXiv: https://doi.org/10.48550/arXiv.2202.10054

---
**Nota metodológica:** Para citar estos trabajos en IEEE use el formato:  
`[1] A. Dosovitskiy et al., "An Image is Worth 16x16 Words," in ICLR, 2021. doi: 10.48550/arXiv.2010.11929`